# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors

Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id`.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

In [ ]:
# Make sure `mlcroissant` and visualization packages are installed
!pip install -U mlcroissant seaborn matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview

Let's review the available record sets and their fields—referencing everything by its `@id`.

In [ ]:
# Find all record set @ids in the dataset
record_sets = []
for record_set in metadata.record_sets:
    print(f"RecordSet name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)

    # List all fields (columns) in each record set
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction

Now, we'll load the records from each record set into a Pandas DataFrame.

All references use the full Croissant `@id` for each record set and field.

In [ ]:
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet {record_set_id}")

if len(record_sets) > 0:
    example_record_set_id = record_sets[0]
    print(f"Columns in record set {example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())
else:
    print('No record sets found in this dataset.')

## 4. Exploratory Data Analysis (EDA)

Now let's process the data: 
- We'll select a numeric field, filter records, normalize it, and group by a key attribute.

**All field and set references use the `@id`.**

In [ ]:
# Pick the main clinical tabular record set (first one by default)
record_set_id = record_sets[0]
df = dataframes[record_set_id].copy()

# Identify numeric fields from the RecordSet
numeric_field_id = None
group_field_id = None
for record_set in metadata.record_sets:
    if record_set.id == record_set_id:
        for field in record_set.fields:
            if field.data_type and ('Integer' in field.data_type or 'Float' in field.data_type or 'Number' in field.data_type):
                if not numeric_field_id:
                    numeric_field_id = field.id
            elif group_field_id is None and field.data_type and ('Text' in field.data_type or 'String' in field.data_type):
                group_field_id = field.id
        break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
else:
    print('No numeric field found.')
if group_field_id:
    print(f"Using group (categorical) field: {group_field_id}")
else:
    print('No suitable grouping field found.')

# Clean up column names (string conversion for robust notebook behavior)
df.columns = [str(col) for col in df.columns]

# Convert the numeric field to numeric if possible
if numeric_field_id and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)   # Example: filter for upper quartile
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by the grouping field if exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)

else:
    print('Cannot perform numeric EDA: No numeric field found in DataFrame.')

## 5. Visualization

We'll visualize the distribution of our selected numeric field, and, if possible, show its relationship with the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load and examine the FAIR² dataset using `mlcroissant`, referencing all entities by `@id` as best practice for reproducibility.
- We explored the available record sets and fields, extracted the data into DataFrames, processed and filtered the data using a numeric `@id`, and visualized some distributions.
- You can further explore the dataset by using other field `@id`s, applying different filters, or combining record sets as suitable for your analysis.